# D1.7 · Drift monitoring

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *Security of AI*

Builds on **[D1.6 · Distinguishing agent from human](https://spbreed.github.io/cyber-commons/lessons/D1.6.html)**.

| | |
|---|---|
| Open-source tooling | promptfoo |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Drift monitoring exists because an agent's behaviour changes **without a code
change**. A new model version, an edited prompt, an added tool — none of these
pass through the change management process built for code, and all of them
invalidate the testing your controls were signed off against.

That is the precise claim: the control was tested against a behaviour that no
longer exists. It has not failed; it is *unevidenced*, which is a different and
more honest state.

Two things are needed:

1. A **signed-off baseline** — what normal looked like when the control passed.
2. A **freshness window** on the control test, derived from how fast the thing
   it tests actually drifts.

E1.7 turns the second into a compliance posture. This lesson produces the signal.

## 2 · Demo — drift across a quarter

In [ ]:
import time
from dataclasses import dataclass, field

now = time.time(); DAY = 86400

@dataclass
class Baseline:
    signed_off: float
    tool_mix: dict
    def compare(self, mix):
        total = sum(mix.values()) or 1
        cur = {k: v/total for k, v in mix.items()}
        keys = set(cur) | set(self.tool_mix)
        tvd = sum(abs(cur.get(k,0) - self.tool_mix.get(k,0)) for k in keys)/2
        return {"drift": round(tvd, 3),
                "new_tools": sorted(set(cur) - set(self.tool_mix)),
                "gone": sorted(set(self.tool_mix) - set(cur))}

base = Baseline(signed_off=now - 90*DAY,
                tool_mix={"read_file": 0.80, "search": 0.15, "write_file": 0.05})

TIMELINE = [
 (now - 90*DAY, "control signed off",     {"read_file": 800, "search": 150, "write_file": 50}),
 (now - 60*DAY, "prompt edited",          {"read_file": 700, "search": 150, "write_file": 150}),
 (now - 30*DAY, "tool added (no PR)",     {"read_file": 500, "search": 120, "write_file": 180,
                                           "run_shell": 200}),
 (now -  5*DAY, "model upgraded by vendor",{"read_file": 300, "search": 100, "write_file": 250,
                                            "run_shell": 350}),
]
print(f"{'when':>8}  {'event':26s}{'drift':>7}  new tools")
print("-" * 66)
for ts, event, mix in TIMELINE:
    d = base.compare(mix)
    print(f"{(now-ts)/DAY:>6.0f}d  {event:26s}{d['drift']:>7.3f}  {d['new_tools']}")

## 3 · Where it breaks — none of these was a code change

In [ ]:
CHANGE_SURFACES = {
 "application code":  ("yes", "PR, review, CI"),
 "agent prompt":      ("no",  "edited in a console"),
 "tool manifest":     ("no",  "config change, no threat-model diff"),
 "model version":     ("no",  "provider-side; you may not be told"),
 "policy (in git)":   ("yes", "if it is in git"),
 "approval settings": ("no",  "a toggle in an admin UI"),
}
print(f"{'surface':20s}{'in change mgmt?':18s}what happens today")
print("-" * 68)
for k, (managed, how) in CHANGE_SURFACES.items():
    print(f"{k:20s}{managed:18s}{how}")
unmanaged = [k for k, (m, _) in CHANGE_SURFACES.items() if m == "no"]
print(f"\n{len(unmanaged)}/{len(CHANGE_SURFACES)} surfaces bypass change management: {unmanaged}")

## 4 · The control — freshness derived from the observed drift rate

In [ ]:
def drift_rate(baseline, timeline):
    """How fast does this agent actually drift? Set the window from the answer."""
    pts = [(ts, baseline.compare(mix)["drift"]) for ts, _, mix in timeline]
    pts.sort()
    span_days = (pts[-1][0] - pts[0][0]) / 86400
    return (pts[-1][1] - pts[0][1]) / max(span_days, 1)

rate = drift_rate(base, TIMELINE)
TOLERANCE = 0.25
window = int(TOLERANCE / rate) if rate > 0 else 365
print(f"observed drift rate  {rate:.5f} TVD/day")
print(f"tolerance            {TOLERANCE}")
print(f"→ freshness window   {window} days "
      f"(a control test older than this is unevidenced, not passing)")

@dataclass
class ControlTest:
    cid: str; passed: bool; tested_at: float; valid_for_days: float
    def state(self, at):
        age = (at - self.tested_at) / 86400
        if age > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

tests = [ControlTest("SB-1", True, now - 90*DAY, window),
         ControlTest("SB-2", True, now - 10*DAY, window),
         ControlTest("DR-1", False, now, window)]
print(f"\n{'control':10s}{'age (d)':>9}{'state':>10}")
print("-" * 30)
for t in tests:
    print(f"{t.cid:10s}{(now-t.tested_at)/DAY:>9.0f}{t.state(now):>10}")
evidenced = sum(t.state(now) == "PASS" for t in tests)
print(f"\ncurrently evidenced: {evidenced}/{len(tests)}")
assert any(t.state(now) == "STALE" for t in tests)

## What you just proved

Drift rises across the quarter from 0.0 at sign-off to roughly 0.35 after the model upgrade, with `run_shell` appearing as a new tool. Four of six change surfaces bypass change management. The observed drift rate yields a freshness window, and the 90-day-old control test is reported STALE rather than passing.

## Your turn

Compute the drift rate for one production agent from three months of telemetry, and set its control freshness window from that number rather than from the audit calendar.

---

**Next → [D1.8 · Threat intel sub-lane](https://spbreed.github.io/cyber-commons/lessons/D1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*